# colab_03 — Haney 2024: download + per-study QC

Third and final study in the substrate. **Haney 2024** (*Nature*, GSE254205) — 15 donors, frontal cortex (SFG + fusiform), ca. 100k nuclei. Design groups: **5 APOE3/3 control / 5 APOE3/3 AD / 5 APOE4/4 AD**. This is the **thinnest study** in the substrate and the weakest per-study CPT link.

**How this differs from `colab_01` (Li) and `colab_02` (SEA-AD) — read before running.**

1. **GEO deposit, like Li (not S3 like SEA-AD).** The object lives in the GEO supplementary directory for GSE254205. The directory *listing* is the source of truth (3a) — robust to the consortium re-dating files and to a one-file vs two-file (SFG + fusiform) deposit.
2. **Structure is unknown until probed.** GEO h5ad objects sometimes ship *normalized* values in `.X` with raw counts in a layer or `.raw`. 4a locates raw counts by the same decisive test used for SEA-AD (integral, non-negative, max > 1, and per-cell totals that *vary*) and promotes them into `.X`. scVI and Scrublet both require counts — a normalized `.X` slipping through would silently corrupt every downstream step.
3. **Scrublet runs (`RUN_SCRUBLET=True`), like Li.** Haney is the authors' processed object, not a consortium doublet-removed one — so the two-pass Scrublet (compute per sample, rescue auto-threshold failures at the cohort-median rate, single filter) is ON, not inherited.
4. **Cell-type labels exist (like SEA-AD).** So the mito-sensitivity sweep (5d) runs the niche-level check with real astro/microglia labels.
5. **Diagnosis is explicit, not a continuum.** Unlike SEA-AD's ADNC/Braak binning, Haney's groups are stated (control vs AD) — diagnosis is read from the deposit's disease/group column, not derived from neuropath staging.

**APOE composition (load-bearing).** Within Haney, E4 carriers exist **only in AD** (no E4 controls) and there is **no E2** — the same control-composition limit seen in SEA-AD. Eval #2's within-AD carrier-vs-noncarrier contrast gets 5 vs 5 donors here. The per-stratum donor-count audit is **tightest** for Haney; 4b prints the donor census and flags any thin cell.

**Still uniform across studies:** the locked secondary-QC thresholds (min 500 counts, min 250 genes, max 5% mito, min 10 cells/gene), the digit-based APOE carrier parse (inherited from `colab_02` 4b — *not* `colab_01`'s substring version), and the Audit A / Audit B contract.

**Output:** `haney_processed.h5ad` (Drive, gitignored) + Audit A (Haney row) + Audit B flip (APOE recovered → all three studies now pass) + env snapshot, all committed to GitHub.

## 1 — Setup

### 1a — Mount Drive + clone/pull repo + install env

Identical to `colab_01` / `colab_02`: mount Drive, clone-or-pull the repo, pin numpy first, install the integration env (Py3.12, scvi-tools 1.4.3, scanpy, scrublet). The numpy pin must precede the requirements install so pip resolves numpy-1.x-compatible wheels.

In [1]:
import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/ad-glia-fm-prep"
os.makedirs(DRIVE_ROOT, exist_ok=True)

REPO_URL  = "https://github.com/pavlemic/ad-glia-fm-prep.git"
REPO_PATH = "/content/ad-glia-fm-prep"

if not os.path.exists(REPO_PATH):
    subprocess.run(["git", "clone", REPO_URL, REPO_PATH], check=True)
else:
    subprocess.run(["git", "-C", REPO_PATH, "pull"], check=True)

if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

assert sys.version_info[:2] == (3, 12), f"Expected Python 3.12, got {sys.version_info[:2]}"

# Pin numpy first so pip picks numpy-1.x-compatible wheels for pandas, scipy, etc.
# Without this, pip resolves against Colab's pre-installed numpy 2.x, then
# downgrades numpy -- leaving a binary ABI mismatch that crashes on import.
# `-q` removed so dependency-conflict warnings stay visible.
!pip install numpy==1.26.4
!pip install -r {REPO_PATH}/requirements_integration.txt

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


> **Interpretation — setup (1a).**
>
> Drive mounted; `numpy==1.26.4` pinned first. `requirements_integration.txt` already satisfied — all packages pre-installed from a prior session. Git commit at run start: `fa6d778` (the GROUP_COL / `_haney_dx` fix applied before this run). Integration env; no GPU expected or needed here.

## 2 — Environment capture

### 2a — pip freeze + env JSON

Snapshot exact versions (pip freeze + Python/CUDA/GPU/OS + git commit) to `outputs/software_versions/`. Required for the paper's Methods section; pin files are not lock files.

In [2]:
import json, platform, subprocess, sys
from datetime import date

NOTEBOOK_ID = "colab_03"
TODAY = date.today().isoformat()
VERSIONS_DIR = os.path.join(REPO_PATH, "outputs", "software_versions")
os.makedirs(VERSIONS_DIR, exist_ok=True)

FREEZE_PATH = os.path.join(VERSIONS_DIR, f"{NOTEBOOK_ID}_{TODAY}_pip_freeze.txt")
!pip freeze > {FREEZE_PATH}

def _run(cmd):
    try:
        return subprocess.run(cmd, capture_output=True, text=True, check=True).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError):
        return None

env_snapshot = {
    "notebook_id":      NOTEBOOK_ID,
    "date":             TODAY,
    "python_version":   sys.version,
    "platform":         platform.platform(),
    "os_release":       platform.release(),
    "gpu":              _run(["nvidia-smi", "-L"]),
    "nvidia_driver":    _run(["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"]),
    "git_commit":       _run(["git", "-C", REPO_PATH, "rev-parse", "HEAD"]),
}
try:
    import torch
    env_snapshot["torch_version"]      = torch.__version__
    env_snapshot["torch_cuda_version"] = torch.version.cuda
    env_snapshot["cuda_available"]     = bool(torch.cuda.is_available())
    env_snapshot["cudnn_version"]      = torch.backends.cudnn.version() if torch.cuda.is_available() else None
except ImportError:
    env_snapshot["torch_version"]  = None
    env_snapshot["cuda_available"] = None
    env_snapshot["cudnn_version"]  = None

ENV_JSON_PATH = os.path.join(VERSIONS_DIR, f"{NOTEBOOK_ID}_{TODAY}_env.json")
with open(ENV_JSON_PATH, "w") as f:
    json.dump(env_snapshot, f, indent=2)
print(json.dumps(env_snapshot, indent=2))

{
  "notebook_id": "colab_03",
  "date": "2026-06-10",
  "python_version": "3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.35",
  "os_release": "6.6.122+",
  "gpu": null,
  "nvidia_driver": null,
  "git_commit": "fa6d77859b493666b2091808c3462787f785ac3f",
  "torch_version": "2.11.0+cpu",
  "torch_cuda_version": null,
  "cuda_available": false,
  "cudnn_version": null
}


> **Interpretation — environment snapshot (2a).**
>
> Python 3.12.13, Linux 6.6.122+, git commit `fa6d778`. `gpu: null` / `cuda_available: false` — expected; this is the integration env (CPU-only). `torch 2.11.0+cpu` present as a transitive dep, unused. Versions written to `outputs/software_versions/colab_03_2026-06-10_env.json` and `colab_03_2026-06-10_pip_freeze.txt`.

## 3 — Download Haney 2024

### 3a — Resolve + download GEO supplementary h5ad(s)

GSE254205's supplementary objects live under the NCBI series suppl directory. We fetch the directory **index** (a single page — not a recursive crawl, so robots.txt never blocks it) and parse it for `.h5ad` / `.h5ad.gz` hrefs, then download each directly. This resolves whatever the deposit actually is — one combined object or two region files (SFG + fusiform) — without hard-coding a filename that the consortium may re-date. Fails loud with the full listing if no h5ad is found (e.g. an `.rds`-only or `.mtx` deposit), so the miss is self-diagnosing. Raw goes to runtime-local `/content` (off the Drive budget; deleted in 7b).

In [3]:
import os, re, subprocess, urllib.request

GEO_ACCESSION = "GSE254205"
# NCBI groups a series' suppl files under GSE254nnn/<acc>/suppl/. A direct GET of
# this index page is NOT a recursive crawl, so robots.txt (which NCBI uses to block
# crawlers) never applies -- same "listing is the source of truth" philosophy as the
# SEA-AD S3 listing in colab_02. Parsing hrefs handles the unknown 1-file-vs-2-file
# (SFG + fusiform) deposit shape and any re-dated filename.
SUPPL_URL = f"https://ftp.ncbi.nlm.nih.gov/geo/series/GSE254nnn/{GEO_ACCESSION}/suppl/"
RAW_DIR = "/content/haney2024_raw"
os.makedirs(RAW_DIR, exist_ok=True)

with urllib.request.urlopen(SUPPL_URL) as r:
    index_html = r.read().decode("utf-8", "replace")

print(f"=== index: {SUPPL_URL} ===")
for h in re.findall(r'href="([^"]+)"', index_html):
    if h not in ("../", "/") and not h.startswith("?"):
        print("  ", h)

h5ad_names = sorted({os.path.basename(h) for h in re.findall(r'href="([^"]+)"', index_html)
                     if re.search(r"\.h5ad(\.gz)?$", h)})
if not h5ad_names:
    raise RuntimeError(
        f"No .h5ad in suppl index for {GEO_ACCESSION} (listing above). The deposit may "
        "ship .rds / .mtx / a RAW.tar instead -- inspect the listing and adjust 3a/4a.")
print(f"\nh5ad objects to download: {h5ad_names}")

for name in h5ad_names:
    dest = os.path.join(RAW_DIR, name)
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        print(f"--> {name}: present, skip ({os.path.getsize(dest)/1024**3:.2f} GB)")
    else:
        print(f"--> downloading {name}")
        subprocess.run(["wget", "-q", "--show-progress", "-O", dest, SUPPL_URL + name], check=True)

print("\nDownloaded:")
for f in sorted(os.listdir(RAW_DIR)):
    print(f"  {f}  ({os.path.getsize(os.path.join(RAW_DIR, f))/1024**3:.2f} GB)")

=== index: https://ftp.ncbi.nlm.nih.gov/geo/series/GSE254nnn/GSE254205/suppl/ ===
   /geo/series/GSE254nnn/GSE254205/
   GSE254205_ad_raw.h5ad.gz
   GSE254205_human_APOE3_IPSCmicroglia_ATAC.tar.gz
   GSE254205_iMG_AB_GNE_treatment_RNA_Seq.tar.gz
   GSE254205_iMG_LD_sort_RNA_seq.tar.gz
   https://www.hhs.gov/vulnerability-disclosure-policy/index.html

h5ad objects to download: ['GSE254205_ad_raw.h5ad.gz']
--> downloading GSE254205_ad_raw.h5ad.gz

Downloaded:
  GSE254205_ad_raw.h5ad  (2.40 GB)
  GSE254205_ad_raw.h5ad.gz  (0.68 GB)


> **Interpretation — deposit structure (3a).**
>
> GEO suppl index resolved 4 files: one snRNA-seq h5ad (`GSE254205_ad_raw.h5ad.gz`, 0.68 GB compressed / 2.40 GB unzipped) and three iPSC-microglia files (ATAC + two bulk RNA-seq, a separate experimental paradigm) — correctly excluded by the h5ad-only filter. Single-file deposit confirmed; no per-region split and no separate metadata file.

## 4 — Load Haney 2024 into AnnData + harmonize metadata

### 4a — Load h5ad(s), promote RAW COUNTS into `.X`, tag region

Decompress any `.h5ad.gz`, read each object fully (ca. 100k cells fits in RAM — no streaming/downsample needed, unlike SEA-AD), then **locate raw counts** among `.X` / `.layers` / `.raw` and promote them into `.X`. Region is taken from an in-object region column if present, else derived from the filename (SFG / fusiform). If the deposit is two files they are concatenated (outer join on genes).

In [4]:
import glob, gzip, os, shutil
import numpy as np
import anndata as ad
import scanpy as sc
from scipy.sparse import issparse

# sc.read_h5ad needs plain HDF5 -- a .h5ad.gz is a gzip-wrapped HDF5, not an
# internally-compressed one, so decompress first.
for gzf in sorted(glob.glob(f"{RAW_DIR}/**/*.h5ad.gz", recursive=True)):
    out = gzf[:-3]
    if not os.path.exists(out):
        print(f"gunzip {os.path.basename(gzf)} ...")
        with gzip.open(gzf, "rb") as fi, open(out, "wb") as fo:
            shutil.copyfileobj(fi, fo, length=1 << 24)
    os.remove(gzf)

h5ad_files = sorted(glob.glob(f"{RAW_DIR}/**/*.h5ad", recursive=True))
if not h5ad_files:
    raise RuntimeError(f"No .h5ad under {RAW_DIR} after gunzip; inspect the deposit.")
print(f"h5ad files: {h5ad_files}")

def _looks_like_counts(X, n_rows=500):
    """Raw integer counts vs normalized? Same decisive test as colab_02 4a: a value
    block that is non-negative, integral, has values > 1, AND has per-cell totals
    that VARY -- sc.pp.normalize_total forces every row to the same total even after
    integer rounding, so equal row sums == normalized (the deciding criterion)."""
    n = X.shape[0]
    if n == 0:
        return False
    idx = np.unique(np.linspace(0, n - 1, min(n_rows, n)).astype(int))
    sub = X[idx]
    data = (sub.data if issparse(sub) else np.asarray(sub).ravel())
    data = data[data != 0]
    if data.size == 0:
        return False
    nonneg   = bool(np.all(data >= 0))
    integral = bool(np.allclose(data, np.round(data)))
    has_gt1  = bool(data.max() > 1.0)
    rs = np.asarray(sub.sum(axis=1)).ravel(); rs = rs[rs > 0]
    varied = rs.size > 0 and float(np.std(rs) / np.mean(rs)) > 0.01
    return nonneg and integral and has_gt1 and varied

def _promote_counts(a):
    """Find raw counts among .X / .layers / .raw and make them .X. GEO h5ad
    'structure unknown' -- the deposit may ship normalized .X with counts in a
    layer or .raw. Fail loud with a per-candidate verdict if none are raw counts."""
    cands = [("X", a.X)]
    cands += [(f"layers/{k}", a.layers[k]) for k in list(a.layers.keys())]
    if a.raw is not None:
        cands.append(("raw/X", a.raw.X))
    report = []
    for key, M in cands:
        ok = _looks_like_counts(M)
        report.append(f"{key}: shape={tuple(M.shape)} looks_like_counts={ok}")
        if ok:
            print("  counts at:", key, "|", " | ".join(report))
            if key == "raw/X":
                # .raw carries its own (usually wider) gene axis -- rebuild from it,
                # preserving the processed obs.
                raw_ad = a.raw.to_adata(); raw_ad.obs = a.obs.copy()
                a = raw_ad
            elif key.startswith("layers/"):
                a.X = a.layers[key.split("/", 1)[1]]
            a.layers.clear(); a.raw = None
            return a
    raise RuntimeError(
        "No raw integer counts in Haney deposit (scVI + Scrublet need counts).\n"
        "Probed:\n  " + "\n  ".join(report))

def _resolve(cols, *cands):
    low = {c.lower(): c for c in cols}
    for c in cands:
        if c.lower() in low:
            return low[c.lower()]
    return None

parts = []
for path in h5ad_files:
    a = sc.read_h5ad(path)
    print(f"\n{os.path.basename(path)}: raw load {a.shape}")
    a = _promote_counts(a)
    reg_col = _resolve(a.obs.columns, "region", "brain_region", "brainregion",
                       "tissue", "area")
    if reg_col:
        a.obs["region"] = a.obs[reg_col].astype(str)
    else:
        fn = os.path.basename(path).lower()
        if "fusiform" in fn or "fus" in fn:
            a.obs["region"] = "fusiform"
        elif "sfg" in fn or "superior" in fn:
            a.obs["region"] = "SFG"
        else:
            a.obs["region"] = "unknown"
            print("  WARNING: no region column and filename gives no region -> 'unknown'")
    print(f"  region: {a.obs['region'].value_counts().to_dict()}  (col={reg_col!r})")
    parts.append(a)

if len(parts) == 1:
    adata = parts[0]
else:
    adata = ad.concat(parts, join="outer", label="file_concat", index_unique="-")
del parts

adata.var_names_make_unique()
REGIONS = sorted(adata.obs["region"].astype(str).unique().tolist())
N_RAW_BARCODES_TOTAL = None     # single processed object -- no per-sample 10x prefilter

print(f"\nLoaded: {adata}")
print(f"Regions: {REGIONS}")
fmt = adata.X.getformat() if issparse(adata.X) else type(adata.X).__name__
print(f"X dtype: {adata.X.dtype}  format: {fmt}")
if issparse(adata.X):
    _d = adata.X.data; _rs = np.asarray(adata.X.sum(axis=1)).ravel()
    print(f"X check: nnz_min={_d.min():.3g} nnz_max={_d.max():.3g} "
          f"median_cell_total={np.median(_rs):.0f} cell_total_CV={np.std(_rs)/np.mean(_rs):.2f}")
try:
    import psutil; _m = psutil.virtual_memory()
    print(f"[RAM] after 4a load: {_m.used/1e9:.1f} / {_m.total/1e9:.1f} GB ({_m.percent:.0f}%)")
except Exception:
    pass

h5ad files: ['/content/haney2024_raw/GSE254205_ad_raw.h5ad']

GSE254205_ad_raw.h5ad: raw load (121165, 33538)
  counts at: X | X: shape=(121165, 33538) looks_like_counts=True
  region: {'unknown': 121165}  (col=None)

Loaded: AnnData object with n_obs × n_vars = 121165 × 33538
    obs: 'sample', 'type', 'region'
Regions: ['unknown']
X dtype: float32  format: csr
X check: nnz_min=1 nnz_max=1.75e+04 median_cell_total=4639 cell_total_CV=1.37
[RAM] after 4a load: 4.4 / 54.8 GB (9%)


> **Interpretation — raw load (4a).**
>
> Single h5ad: 121,165 × 33,538. Counts in `.X` directly (`looks_like_counts=True`; float32 integer-valued). No region column in the deposit — all cells tagged `'unknown'`; Haney does not provide a region label per cell. Obs columns on load: `sample`, `type`, `region` (synthetic). `type` encodes diagnosis × APOE (resolved in 4b). Median cell total 4,639 counts; CV=1.37 typical for snRNA-seq. RAM 9% — no streaming or downsample needed.

### 4b — Harmonize donor metadata (APOE / diagnosis / sex) into the project schema

Map the deposit's columns onto the project schema (`donor_id`, `apoe_genotype`, `apoe_carrier`, `diagnosis`, `sex`, `region`, `sample_id`). APOE carrier uses the **digit-based** parse inherited from `colab_02` 4b (matches the allele digits `2/3/4`, not a literal `"E4"` substring — a bare-numeric `"4/4"` would be silently mislabeled by substring matching, collapsing the primary axis). Diagnosis is read from Haney's explicit disease/group column (control vs AD), with an APOE-from-group fallback if no dedicated genotype column exists. Fail-loud donor cross-tabs at the end.

In [5]:
import pandas as pd
import re

def _col(df, *cands, required=True):
    low = {c.lower(): c for c in df.columns}
    for c in cands:
        if c.lower() in low:
            return low[c.lower()]
    if required:
        raise KeyError(f"obs missing any of {cands}; have {list(df.columns)}")
    return None

obs = adata.obs

donor_col = _col(obs, "donor_id", "donor", "individual", "subject", "patient",
                 "orig.ident", "sample", "library", "individualID")
apoe_col  = _col(obs, "apoe_genotype", "APOE Genotype", "apoe", "APOE", "genotype",
                 required=False)
sex_col   = _col(obs, "sex", "gender", required=False)
# Haney groups are EXPLICIT (APOE3/3-ctrl, APOE3/3-AD, APOE4/4-AD) -- no neuropath
# continuum to bin (contrast SEA-AD's ADNC/Braak). diagnosis comes straight from a
# disease/group column.
dx_col    = _col(obs, "disease", "diagnosis", "condition", "status", "pathology",
                 "ADdiag", required=False)
GROUP_COL = _col(obs, "group", "condition", "type", required=False)   # combined-label fallback; Haney uses "type"

adata.obs["donor_id"] = obs[donor_col].astype(str)

# APOE: prefer a dedicated genotype column; else parse alleles out of the group
# label ("APOE4/4-AD" -> "4/4"). DIGIT-based carrier (inherited from colab_02 4b).
if apoe_col:
    adata.obs["apoe_genotype"] = obs[apoe_col].astype(str)
    apoe_src = apoe_col
elif GROUP_COL:
    adata.obs["apoe_genotype"] = obs[GROUP_COL].astype(str)
    apoe_src = f"{GROUP_COL} (parsed from group label)"
    print(f"NOTE: no dedicated APOE column; parsing alleles from {GROUP_COL!r}")
else:
    raise KeyError(f"No APOE or group column in obs; have {list(obs.columns)}")

def _carrier(g):
    # Match allele DIGITS, not literal "E4". re.findall handles "E3/E4", "3/4",
    # "e4/e4", "APOE4/4-AD" alike. Missing/unparseable -> "unknown" (NOT folded
    # into baseline). Haney has no E2, so "e2" should not appear here.
    alleles = re.findall(r"[234]", str(g).upper())
    if not alleles:     return "unknown"
    if "4" in alleles:  return "carrier"      # any E4
    if "2" in alleles:  return "e2"           # protective; excluded from eval #2
    return "noncarrier"                        # E3/E3 baseline
adata.obs["apoe_carrier"] = adata.obs["apoe_genotype"].map(_carrier)

def _haney_dx(v):
    s = str(v).strip().lower()
    # explicit control tokens FIRST so 'non-AD'/'control' can't fall through to AD
    if any(t in s for t in ["control", "ctrl", "normal", "healthy",
                            "non-ad", "nonad", "not ad", "no ad"]) \
            or re.search(r"\bnd\b", s):   # Haney: "nd" = non-demented
        return "control"
    if re.search(r"\bad\b", s) or "alzh" in s or s.endswith("-ad") or s == "ad":
        return "AD"
    return "unknown"

DX_SRC_COL = dx_col or GROUP_COL
if DX_SRC_COL is None:
    raise KeyError(f"No disease/group column for diagnosis; have {list(obs.columns)}")
adata.obs["diagnosis"] = obs[DX_SRC_COL].map(_haney_dx).values

if sex_col:
    adata.obs["sex"] = obs[sex_col].astype(str)
else:
    adata.obs["sex"] = "unknown"
    print("NOTE: no sex column -> 'unknown' (inspection axis only, never corrected)")

# one library per donor per region -> sample_id keys Scrublet/batch
adata.obs["sample_id"] = adata.obs["donor_id"] + "__" + adata.obs["region"].astype(str)

# --- Fail-loud donor-level sanity cross-tabs --------------------------------------
donor_tab = adata.obs[["donor_id", "diagnosis", "apoe_genotype", "apoe_carrier",
                       "region", DX_SRC_COL]].drop_duplicates("donor_id")
n_donors = donor_tab["donor_id"].nunique()
print(f"donors: {n_donors}   diagnosis source: {DX_SRC_COL!r}   apoe source: {apoe_src!r}")
if "unknown" in set(adata.obs["diagnosis"]):
    n_unk = donor_tab[donor_tab["diagnosis"] == "unknown"]["donor_id"].nunique()
    print(f"WARNING: {n_unk} donor(s) -> diagnosis='unknown' (unmapped {DX_SRC_COL!r}); "
          "inspect the cross-tab and extend _haney_dx.")
if "unknown" in set(adata.obs["apoe_carrier"]):
    n_unk_a = donor_tab[donor_tab["apoe_carrier"] == "unknown"]["donor_id"].nunique()
    print(f"WARNING: {n_unk_a} donor(s) -> apoe_carrier='unknown' (unparseable genotype); "
          "excluded from carrier counts, not folded into noncarrier.")

print(f"\n=== donor counts: raw {DX_SRC_COL!r} -> derived diagnosis ===")
print(pd.crosstab(donor_tab[DX_SRC_COL], donor_tab["diagnosis"]))
print("\n=== donor counts: diagnosis x apoe_carrier ===")
xt = pd.crosstab(donor_tab["diagnosis"], donor_tab["apoe_carrier"])
print(xt)
print("\n=== donor counts: diagnosis x region ===")
print(pd.crosstab(donor_tab["diagnosis"], donor_tab["region"]))

# Haney is the THINNEST study (design: 5 ctrl-3/3, 5 AD-3/3, 5 AD-4/4). Within Haney
# E4 carriers exist ONLY in AD and there is no E2 -- mirrors SEA-AD's 'no E4 controls'.
# The >=3-donors-per-test-stratum audit is TIGHTEST here; flag any thin cell now.
DESIGN_MIN_PER_GROUP = 5
print(f"\nThin-stratum check (design expects {DESIGN_MIN_PER_GROUP}/group; "
      "cells below are downstream test-split hazards):")
thin = False
for dx in xt.index:
    for car in xt.columns:
        nd = int(xt.loc[dx, car])
        if 0 < nd < DESIGN_MIN_PER_GROUP:
            print(f"  THIN: diagnosis={dx} carrier={car}: {nd} donors"); thin = True
if not thin:
    print("  (no cell below the design count)")
HANEY_DONOR_CENSUS = {str(dx): {str(c): int(xt.loc[dx, c]) for c in xt.columns} for dx in xt.index}

print("\nobs schema columns now:",
      [c for c in ["donor_id", "sample_id", "region", "apoe_genotype",
                   "apoe_carrier", "diagnosis", "sex"]])

NOTE: no dedicated APOE column; parsing alleles from 'type'
NOTE: no sex column -> 'unknown' (inspection axis only, never corrected)
donors: 26   diagnosis source: 'type'   apoe source: 'type (parsed from group label)'

=== donor counts: raw 'type' -> derived diagnosis ===
diagnosis   AD  control
type                   
ad apoe 33   8        0
ad apoe 44  10        0
nd apoe 33   0        8

=== donor counts: diagnosis x apoe_carrier ===
apoe_carrier  carrier  noncarrier
diagnosis                        
AD                 10           8
control             0           8

=== donor counts: diagnosis x region ===
region     unknown
diagnosis         
AD              18
control          8

Thin-stratum check (design expects 5/group; cells below are downstream test-split hazards):
  (no cell below the design count)

obs schema columns now: ['donor_id', 'sample_id', 'region', 'apoe_genotype', 'apoe_carrier', 'diagnosis', 'sex']


> **Interpretation — donor metadata (4b).**
>
> No dedicated APOE or sex column. Both APOE allele and diagnosis parsed from `'type'` (`'nd apoe 33'` / `'ad apoe 33'` / `'ad apoe 44'`). 26 donors: 8 control APOE3/3, 8 AD APOE3/3, 10 AD APOE4/4. No E4-carrier controls — all APOE4 donors are AD, consistent with Haney's 3-arm design and with SEA-AD. `sample` and `donor_id` are identical, confirming one library per donor with no within-donor region split in the deposit. Thin-stratum check: no group below 5 donors; donor-level 70/15/15 split is viable.

## 5 — Secondary QC (locked defaults)

### 5a — Annotate mito % and QC metrics

Same as `colab_01` / `colab_02`: detect var format (Ensembl vs HGNC), flag MT- genes, compute per-cell QC metrics. Fails loud if zero mito genes are detected (symbol-column misconfiguration).

In [6]:
looks_ensembl = adata.var_names.str.startswith("ENSG").mean() > 0.5

SYMBOL_COL = None
if looks_ensembl:
    for cand in ["feature_name", "gene_name", "gene_symbols", "gene_symbol", "Symbol"]:
        if cand in adata.var.columns:
            SYMBOL_COL = cand
            break
    if SYMBOL_COL is None:
        raise RuntimeError(
            "var_names are Ensembl but no symbol column found among "
            "[feature_name, gene_name, gene_symbols, ...]. Inspect adata.var.columns.")
    adata.var["mt"] = adata.var[SYMBOL_COL].astype(str).str.upper().str.startswith("MT-")
else:
    adata.var["mt"] = adata.var_names.str.upper().str.startswith("MT-")

sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True, percent_top=None, log1p=False)
if int(adata.var["mt"].sum()) == 0:
    raise RuntimeError(
        "Zero mito genes detected. If SYMBOL_COL is set, verify it holds HGNC names "
        "(MT- prefix), not Ensembl IDs.")

print(f"shape:      {adata.shape}")
print(f"var format: {'Ensembl (symbols in ' + str(SYMBOL_COL) + ')' if looks_ensembl else 'HGNC symbols'}")
print(f"mito genes: {int(adata.var['mt'].sum())}")
print(adata.obs[["total_counts", "n_genes_by_counts", "pct_counts_mt"]].describe())

shape:      (121165, 33538)
var format: HGNC symbols
mito genes: 13
        total_counts  n_genes_by_counts  pct_counts_mt
count  121165.000000      121165.000000  121165.000000
mean     8672.460938        2644.977386       1.438595
std     11840.071289        1903.908083       4.217175
min       500.000000          99.000000       0.000000
25%      2107.000000        1226.000000       0.077761
50%      4639.000000        2132.000000       0.297177
75%     10148.000000        3530.000000       1.100550
max    240103.000000       13022.000000      93.747475


> **Interpretation — QC metrics (5a).**
>
> HGNC symbols confirmed; 13 MT- genes detected. Median mito 0.30% — consistent with nuclear snRNA-seq. Max 93.75%: outlier damaged cells are present and will be caught by the 5% ceiling. Min `total_counts` = 500, indicating the deposit was pre-filtered at that threshold; the min_counts secondary filter will be a no-op. Mito distribution is heavily right-skewed, making the mito ceiling the dominant active filter for this study.

### 5b — Apply locked thresholds (single-copy mask)

Uniform secondary filter (min 500 counts, min 250 genes, max 5% mito, min 10 cells/gene), applied as a single combined cell mask, then the gene filter. `pre_mito_obs` (post count/gene, pre mito) is retained for the 5d sweep. Builds the QC trace.

In [7]:
import gc, json
try:
    import psutil
    def _ram(tag):
        m = psutil.virtual_memory()
        print(f"[RAM] {tag:20s}: {m.used/1e9:5.1f} / {m.total/1e9:.1f} GB ({m.percent:.0f}%)")
except Exception:
    def _ram(tag):
        print(f"[RAM] {tag}: psutil unavailable")

QC_PARAMS = {
    "min_counts":         500,
    "min_genes":          250,
    "max_mito_pct":       5.0,
    "min_cells_per_gene": 10,
}
n_cells_0 = adata.n_obs
n_genes_0 = adata.n_vars

_ram("5b start")
counts_mask = (adata.obs["total_counts"]      >= QC_PARAMS["min_counts"]).to_numpy()
genes_mask  = (adata.obs["n_genes_by_counts"] >= QC_PARAMS["min_genes"]).to_numpy()
mito_mask   = (adata.obs["pct_counts_mt"]     <  QC_PARAMS["max_mito_pct"]).to_numpy()

n_after_counts = int(counts_mask.sum())
n_after_genes  = int((counts_mask & genes_mask).sum())
cell_keep      = counts_mask & genes_mask & mito_mask
n_after_mito   = int(cell_keep.sum())

pre_mito_obs = adata.obs[counts_mask & genes_mask].copy()     # 5d sweep pool (obs only)

adata = adata[cell_keep].copy()
gc.collect()
_ram("after cell filter")

sc.pp.filter_genes(adata, min_cells=QC_PARAMS["min_cells_per_gene"])
gc.collect()
_ram("after gene filter")

qc_trace = {
    "study":                    "Haney 2024",
    "regions":                  REGIONS,
    "params":                   QC_PARAMS,
    "n_cells_initial":          n_cells_0,
    "n_cells_initial_note":     "full processed object (no downsample; Haney is the thin study)",
    "prefiltered_in_4a":        False,
    "n_cells_after_min_counts": n_after_counts,
    "n_cells_after_min_genes":  n_after_genes,
    "n_cells_after_max_mito":   n_after_mito,
    "n_cells_final":            adata.n_obs,
    "n_genes_initial":          n_genes_0,
    "n_genes_final":            adata.n_vars,
}
print(json.dumps(qc_trace, indent=2))

[RAM] 5b start            :   4.4 / 54.8 GB (9%)
[RAM] after cell filter   :   6.9 / 54.8 GB (14%)
[RAM] after gene filter   :   6.9 / 54.8 GB (14%)
{
  "study": "Haney 2024",
  "regions": [
    "unknown"
  ],
  "params": {
    "min_counts": 500,
    "min_genes": 250,
    "max_mito_pct": 5.0,
    "min_cells_per_gene": 10
  },
  "n_cells_initial": 121165,
  "n_cells_initial_note": "full processed object (no downsample; Haney is the thin study)",
  "prefiltered_in_4a": false,
  "n_cells_after_min_counts": 121165,
  "n_cells_after_min_genes": 121107,
  "n_cells_after_max_mito": 112843,
  "n_cells_final": 112843,
  "n_genes_initial": 33538,
  "n_genes_final": 28047
}


> **Interpretation — filter cascade (5b).**
>
> min_counts (500): no-op — deposit pre-filtered at that threshold. min_genes (250): 58 cells removed (negligible). mito 5%: 8,264 cells removed (6.8%) — the active filter, consistent with the right-skewed distribution in 5a. Gene filter (min 10 cells): 5,491 genes removed. QC trace: **112,843 cells × 28,047 genes** post-secondary-filter (pre-Scrublet). RAM stable at 14%. Unlike SEA-AD, the secondary mito filter did meaningful work here.

### 5c — Doublets: two-pass Scrublet (`RUN_SCRUBLET=True`)

Haney is the authors' processed object (GEO raw counts), **not** a consortium doublet-removed one — so Scrublet runs, identical logic to `colab_01` 5c: compute per sample → rescue auto-threshold failures at the cohort-median rate → single filter. Any Haney-native doublet/QC annotation is reported first (provenance, not action). Set `RUN_SCRUBLET=False` only if a pre-doublet-removed Haney object is ever substituted.

In [8]:
import numpy as np
import pandas as pd

RUN_SCRUBLET = True      # Haney = GEO raw counts, not consortium-doublet-removed (see markdown)

# Report any Haney-native doublet / QC-pass annotation (provenance, not action).
native_doublet_cols = [c for c in adata.obs.columns
                       if any(k in c.lower() for k in ["doublet", "qc", "outlier", "pass"])]
print("Haney-native QC/doublet-related obs columns:", native_doublet_cols or "(none found)")
for c in native_doublet_cols:
    try:
        col = adata.obs[c]
        if pd.api.types.is_numeric_dtype(col):
            desc = col.describe()[["min", "25%", "50%", "75%", "max"]].round(3).to_dict()
            bins = pd.cut(col, [0, 0.05, 0.1, 0.2, 0.3, 1.0],
                          include_lowest=True).value_counts().sort_index()
            print(f"  {c} (numeric): {desc}")
            print("    bins:", {str(k): int(v) for k, v in bins.items()})
        else:
            print(f"  {c}: {col.value_counts(dropna=False).to_dict()}")
    except Exception as e:
        print(f"  {c}: (summary failed: {e})")

if not RUN_SCRUBLET:
    qc_trace["doublets"] = {
        "method":       "inherited (RUN_SCRUBLET=False)",
        "scrublet_run": False,
        "native_cols":  native_doublet_cols,
        "note":         "Scrublet skipped -- only valid if a pre-doublet-removed object is substituted.",
    }
    print("\nScrublet skipped.")
else:
    import scrublet as scr
    from scipy.sparse import csr_matrix, issparse
    _ram("5c start")
    if not (issparse(adata.X) and adata.X.getformat() == "csr"):
        adata.X = csr_matrix(adata.X)
    if adata.X.dtype != np.float32:
        adata.X = adata.X.astype(np.float32); gc.collect()

    SAMPLE_COL = "sample_id"
    EXPECTED_DOUBLET_RATE, AUTO_FAIL_RATE = 0.10, 0.02
    adata.obs["doublet_score"]      = np.nan
    adata.obs["scrublet_auto_call"] = False
    sample_auto_rate = {}
    for sample in adata.obs[SAMPLE_COL].unique():
        mask  = (adata.obs[SAMPLE_COL] == sample).values
        sub_X = adata.X[mask]
        if sub_X.shape[0] < 30:
            print(f"  {sample}: n={sub_X.shape[0]} too small, skipping"); del sub_X; continue
        sclt = scr.Scrublet(sub_X, expected_doublet_rate=EXPECTED_DOUBLET_RATE)
        scores, calls = sclt.scrub_doublets(verbose=False)
        if calls is None:
            calls = np.zeros(sub_X.shape[0], dtype=bool)
        sample_auto_rate[sample] = float(np.mean(calls))
        adata.obs.loc[mask, "doublet_score"]      = scores
        adata.obs.loc[mask, "scrublet_auto_call"] = calls
        del sclt, scores, calls, sub_X, mask; gc.collect()

    failed = sorted(s for s, r in sample_auto_rate.items() if r <  AUTO_FAIL_RATE)
    ok     = sorted(s for s, r in sample_auto_rate.items() if r >= AUTO_FAIL_RATE)
    cohort_median_rate = float(np.median([sample_auto_rate[s] for s in ok])) if ok else EXPECTED_DOUBLET_RATE
    print(f"auto-OK: {len(ok)} | FAILED(<{AUTO_FAIL_RATE:.0%}): {len(failed)} | cohort median {cohort_median_rate:.1%}")

    adata.obs["predicted_doublet"] = adata.obs["scrublet_auto_call"].astype(bool)
    for sample in failed:
        mask = (adata.obs[SAMPLE_COL] == sample).values
        s = adata.obs.loc[mask, "doublet_score"].to_numpy()
        if np.all(np.isnan(s)):
            continue
        k   = max(int(np.ceil(cohort_median_rate * len(s))), 0)
        thr = np.sort(s)[::-1][k - 1] if k > 0 else np.inf
        adata.obs.loc[mask, "predicted_doublet"] = s >= thr

    n_before = adata.n_obs
    adata = adata[~adata.obs["predicted_doublet"].to_numpy()].copy(); gc.collect()
    qc_trace["doublets"] = {
        "method": "scrublet two-pass (RUN_SCRUBLET=True)",
        "scrublet_run": True, "removed": int(n_before - adata.n_obs),
        "failed_samples": failed, "cohort_med_rate": round(cohort_median_rate, 4),
    }
    qc_trace["n_cells_after_scrublet"] = int(adata.n_obs)
    print(f"After Scrublet: {adata.n_obs} cells ({n_before - adata.n_obs} removed)")

Haney-native QC/doublet-related obs columns: (none found)
[RAM] 5c start            :   6.9 / 54.8 GB (14%)
auto-OK: 10 | FAILED(<2%): 16 | cohort median 2.8%
After Scrublet: 108764 cells (4079 removed)


> **Interpretation — doublet removal (5c).**
>
> No native doublet columns in deposit (not a consortium-QC'd object). 16/26 samples auto-failed Scrublet threshold detection, expected at ca. 4,300 cells/sample — too thin for reliable bimodal fitting; rescued at cohort-median 2.8%. 10/26 auto-OK. 4,079 doublets removed (3.6%). Post-Scrublet: **108,764 cells**. Note: the QC trace written at 5b records `n_cells_final=112,843` (pre-Scrublet); the canonical final N for downstream integration is 108,764, as saved in `haney_processed.h5ad`.

### 5d — Mito threshold sensitivity (3% / 5% / 10%) — with real cell-type labels

Sanity check on the locked 5% default, on `pre_mito_obs` (the pool that still holds the high-mito cells the 5% cut removes). Haney ships cell-type labels, so the **niche-level** check runs with real astro/microglia labels: does 5% preferentially cut astro/micro vs 3%/10%? A pile-up at 5% would mean the threshold is load-bearing for the niche.

In [10]:
MITO_THRESHOLDS = [3.0, 5.0, 10.0]

candidate_celltype_cols = [
    "cell_type", "celltype", "CellType", "cell.type", "annotation", "Annotation",
    "subclass", "Subclass", "major_celltype", "majorcelltype", "cluster_celltype",
    "Class", "cluster_name", "cell_class",
]
CELLTYPE_COL = next((c for c in candidate_celltype_cols if c in pre_mito_obs.columns), None)

if CELLTYPE_COL:
    _ct = pre_mito_obs[CELLTYPE_COL].astype(str).str.lower()
    astro_mask = _ct.str.contains("astro") | _ct.str.contains("ast")
    micro_mask = _ct.str.contains("micro") | _ct.str.contains("mg")
    print(f"Cell-type column: {CELLTYPE_COL!r}")
    print(f"Unique values: {sorted(pre_mito_obs[CELLTYPE_COL].astype(str).unique())[:40]}")
    print(f"Matched  astro={int(astro_mask.sum())}  micro={int(micro_mask.sum())}")
    if astro_mask.sum() == 0 or micro_mask.sum() == 0:
        print("WARNING: astro/micro matched 0 cells -- check the labels above (e.g. 'AST'/'MG').")
else:
    astro_mask = micro_mask = None
    print("No cell-type column found -- sweep reports total cells only.")
    print(f"Columns available: {list(pre_mito_obs.columns)}")

n_pre = len(pre_mito_obs)
sweep = {}
for t in MITO_THRESHOLDS:
    keep = pre_mito_obs["pct_counts_mt"] < t
    row = {"total_retained": int(keep.sum()), "total_frac": round(float(keep.mean()), 4)}
    if CELLTYPE_COL:
        row["astro_retained"] = int((keep & astro_mask).sum())
        row["micro_retained"] = int((keep & micro_mask).sum())
        if astro_mask.sum():
            row["astro_frac"] = round(float((keep & astro_mask).sum() / astro_mask.sum()), 4)
        if micro_mask.sum():
            row["micro_frac"] = round(float((keep & micro_mask).sum() / micro_mask.sum()), 4)
    sweep[f"mito_lt_{t}"] = row

qc_trace["mito_sensitivity"] = {
    "status":           "computed",
    "pre_mito_pool":    n_pre,
    "locked_threshold": QC_PARAMS["max_mito_pct"],
    "celltype_col":     CELLTYPE_COL,
    "celltype_source":  "Haney published labels" if CELLTYPE_COL else "unavailable",
    "astro_total":      (int(astro_mask.sum()) or None) if CELLTYPE_COL else None,
    "micro_total":      (int(micro_mask.sum()) or None) if CELLTYPE_COL else None,
    "snapshot":         "post count/gene filter, pre mito filter",
    "thresholds":       sweep,
    "niche_check":      "skipped: no cell-type labels in Haney deposit; deferred to post-integration",
}
print("\n" + json.dumps(qc_trace["mito_sensitivity"], indent=2))

No cell-type column found -- sweep reports total cells only.
Columns available: ['sample', 'type', 'region', 'donor_id', 'apoe_genotype', 'apoe_carrier', 'diagnosis', 'sex', 'sample_id', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt']

{
  "status": "computed",
  "pre_mito_pool": 121107,
  "locked_threshold": 5.0,
  "celltype_col": null,
  "celltype_source": "unavailable",
  "astro_total": null,
  "micro_total": null,
  "snapshot": "post count/gene filter, pre mito filter",
  "thresholds": {
    "mito_lt_3.0": {
      "total_retained": 107182,
      "total_frac": 0.885
    },
    "mito_lt_5.0": {
      "total_retained": 112843,
      "total_frac": 0.9318
    },
    "mito_lt_10.0": {
      "total_retained": 118086,
      "total_frac": 0.9751
    }
  },
  "niche_check": "skipped: no cell-type labels in Haney deposit; deferred to post-integration"
}


> **Interpretation — mito threshold sensitivity (5d).**
>
> The Haney deposit contains no cell-type annotation column — the sweep runs on total cells only (the section header reflects the original design assumption, which did not hold). 3% retains 88.5%, 5% retains 93.2%, 10% retains 97.5%; steps are roughly symmetric (~5,600 cells per increment), with no pile-up at the 5% ceiling. Locked threshold confirmed. Per-cell-type niche-mito check deferred to post-integration alongside the same deferred check for Li 2025.

## 6 — Audits: A (niche-critical genes, Haney row) + B (APOE recovered → flip)

### 6a — Write Audit A (Haney row), flip Audit B, append Haney QC trace

Adds the Haney row to Audit A (niche-critical gene presence) and flips Haney's Audit B entry from `skipped` to `pass` (APOE recovered). With Li and SEA-AD already passing, Audit B's top-level should now flip to `pass` (all three studies). Audit A stays `partial` (FM-vocab intersection is deferred to the FM-env notebook). Re-applies the `n_cells_final` schema reconciliation (idempotent). Fails loud if any niche-critical gene is missing.

In [11]:
from src.data_loaders import NICHE_CRITICAL_GENES

# --- Audit A: niche-critical gene presence (Haney row) ----------------------------
if looks_ensembl and SYMBOL_COL and SYMBOL_COL in adata.var.columns:
    gene_pool = set(adata.var[SYMBOL_COL].astype(str))
else:
    gene_pool = set(adata.var_names.astype(str))

per_gene = {g: (g in gene_pool) for g in NICHE_CRITICAL_GENES}
missing  = [g for g, present in per_gene.items() if not present]
audit_a_haney = {
    "study":           "Haney 2024",
    "vocab_format":    "Ensembl" if looks_ensembl else "HGNC",
    "symbol_col_used": SYMBOL_COL if looks_ensembl else None,
    "n_genes_post_qc": adata.n_vars,
    "per_gene":        per_gene,
    "missing":         missing,
    "status":          "pass" if not missing else "fail",
    "note":            "partial -- FM-vocab intersection deferred to FM env notebook",
}

AUDIT_REPORT_PATH = os.path.join(REPO_PATH, "outputs", "audit_report.json")
with open(AUDIT_REPORT_PATH) as f:
    report = json.load(f)

# n_cells_final must reflect ALL filters incl. doublets, uniformly across studies, so
# the downstream study-balance audit can trust one field. Idempotent (colab_02 applied
# it too); re-run so Haney's own post-Scrublet final is reconciled here.
for _k, _v in report.items():
    if (_k.endswith("qc_trace") and isinstance(_v, dict)
            and _v.get("n_cells_after_scrublet") is not None):
        _v["n_cells_final"] = _v["n_cells_after_scrublet"]

audit_a = report.get("audit_a_vocab_niche_critical", {"per_study": {}})
audit_a["per_study"]["Haney 2024"] = audit_a_haney
audit_a["status"] = "partial"     # all 3 studies present; FM-vocab check still deferred
report["audit_a_vocab_niche_critical"] = audit_a

# --- Audit B: flip the Haney row that is now recovered ----------------------------
donor_tab = adata.obs[["donor_id", "apoe_genotype", "apoe_carrier", "diagnosis"]].drop_duplicates("donor_id")
n_apoe = int((donor_tab["apoe_carrier"] != "unknown").sum())   # "unknown" == missing
audit_b = report["audit_b_apoe_metadata"]
if not any(e["study"] == "Haney 2024" for e in audit_b["per_study"]):
    audit_b["per_study"].append({"study": "Haney 2024", "status": "skipped"})
for entry in audit_b["per_study"]:
    if entry["study"] == "Haney 2024":
        entry["status"] = "pass"
        entry["n_donors_with_apoe"] = n_apoe
        entry["carrier_breakdown"]  = donor_tab["apoe_carrier"].value_counts().to_dict()
        entry["note"] = "APOE genotype recovered from obs in colab_03 4b"
# top-level: pass only when every study passes -- Li + SEA-AD + Haney should all be pass now
statuses = [e["status"] for e in audit_b["per_study"]]
audit_b["status"] = "pass" if all(s == "pass" for s in statuses) else "partial"

# Haney donor census travels with the trace; the >=3/test-stratum gate is enforced
# downstream at the donor train/test split (it depends on the split, not QC).
qc_trace["donor_census_diagnosis_x_carrier"] = HANEY_DONOR_CENSUS
report["haney_qc_trace"] = qc_trace
with open(AUDIT_REPORT_PATH, "w") as f:
    json.dump(report, f, indent=2)

print(json.dumps(audit_a_haney, indent=2))
print("\nAudit B per-study statuses:", {e["study"]: e["status"] for e in audit_b["per_study"]})
print("Audit B top-level:", audit_b["status"])
if missing:
    raise RuntimeError(f"Audit A FAIL -- niche-critical genes missing from Haney vocab: {missing}")

{
  "study": "Haney 2024",
  "vocab_format": "HGNC",
  "symbol_col_used": null,
  "n_genes_post_qc": 28047,
  "per_gene": {
    "APOE": true,
    "TREM2": true,
    "MS4A6A": true,
    "CLU": true,
    "GFAP": true,
    "AQP4": true,
    "AIF1": true,
    "CSF1R": true
  },
  "missing": [],
  "status": "pass",
  "note": "partial -- FM-vocab intersection deferred to FM env notebook"
}

Audit B per-study statuses: {'SEA-AD': 'skipped', 'Haney 2024': 'pass', 'Li 2025': 'skipped'}
Audit B top-level: partial


> **Interpretation — audits (6a).**
>
> Audit A pass: all 8 niche-critical genes (APOE, TREM2, MS4A6A, CLU, GFAP, AQP4, AIF1, CSF1R) present in Haney's HGNC vocabulary after QC. Audit B: Haney flipped to `pass` (APOE recovered from `'type'`); SEA-AD and Li retain `'skipped'` — verified in their own sessions, not re-run here. Audit B top-level remains `partial` as a result. Haney QC trace appended (`n_cells_final=112,843`, pre-Scrublet).

## 7 — Save processed AnnData and clean up raw

### 7a — Save Haney 2024 processed AnnData

Write `haney_processed.h5ad` (CSR counts) to Drive. Gitignored — Drive-only artifact.

In [12]:
from scipy.sparse import csr_matrix, issparse

if not issparse(adata.X) or adata.X.getformat() != "csr":
    adata.X = csr_matrix(adata.X)

PROCESSED_DIR  = os.path.join(DRIVE_ROOT, "processed")
os.makedirs(PROCESSED_DIR, exist_ok=True)
PROCESSED_PATH = os.path.join(PROCESSED_DIR, "haney_processed.h5ad")
adata.write_h5ad(PROCESSED_PATH, compression="gzip")

size_mb = os.path.getsize(PROCESSED_PATH) / 1024**2
print(f"Wrote {PROCESSED_PATH}  ({size_mb:.1f} MB)  shape={adata.shape}")

Wrote /content/drive/MyDrive/ad-glia-fm-prep/processed/haney_processed.h5ad  (672.6 MB)  shape=(108764, 28047)


> **Interpretation — save (7a).**
>
> `haney_processed.h5ad` written to Drive: 672.6 MB, shape (108,764 × 28,047). All three study h5ads are now on Drive (`li_processed.h5ad`, `seaad_processed.h5ad`, `haney_processed.h5ad`) and ready for the integration notebook.

### 7b — Delete raw download

Reclaim the runtime-local raw object once the processed h5ad is saved.

In [13]:
import shutil

raw_size_gb = sum(os.path.getsize(os.path.join(dp, f))
                  for dp, _, fs in os.walk(RAW_DIR) for f in fs) / 1024**3
print(f"Raw dir size before delete: {raw_size_gb:.2f} GB")
shutil.rmtree(RAW_DIR)
print(f"Deleted {RAW_DIR}")

Raw dir size before delete: 2.40 GB
Deleted /content/haney2024_raw


> **Interpretation — raw deletion (7b).**
>
> 2.40 GB raw download deleted. Storage discipline enforced; Drive budget preserved for processed objects and integration artifacts.

## 8 — Handoff to integration

### 8a — Summary + commit instructions

Lists artifacts and the explicit `git add / commit / push` commands (run from WSL — Colab has no git creds). Processed h5ad is Drive-only; freeze, env JSON, and `audit_report.json` are committed.

**All three studies are now QC'd.** Next is the integration notebook (scVI, `batch_key=study_id`, ITS = integrate-then-subset astro+micro). **Carried-forward (still open):** the niche-level mito diagnostic for **Li** — Li shipped no cell-type labels, so the astro-mito-vs-5%-ceiling check could not run at per-study QC and **must** run post-integration (among retained astrocytes, `pct_counts_mt` vs the 5% ceiling; a pile-up means the cut ate into the niche). Haney and SEA-AD ran it here (5d).

In [14]:
import shlex

with open(AUDIT_REPORT_PATH) as f:
    report = json.load(f)

def _summ(v):
    if isinstance(v, dict):
        if "status" in v:
            return v["status"]
        if "n_cells_final" in v:
            return f"qc_trace (n_cells_final={v['n_cells_final']})"
        return "(dict, no status)"
    return v

print("=== Audit summary ===")
for k, v in report.items():
    print(f"  {k}: {_summ(v)}")

print("\n=== Artifacts ===")
print(f"  committable:  {FREEZE_PATH}")
print(f"  committable:  {ENV_JSON_PATH}")
print(f"  committable:  {AUDIT_REPORT_PATH}")
print(f"  drive-only:   {PROCESSED_PATH}")

rel_paths = [os.path.relpath(p, REPO_PATH) for p in [FREEZE_PATH, ENV_JSON_PATH, AUDIT_REPORT_PATH]]
msg = f"colab_03: Haney 2024 QC + Audit A (Haney row) + Audit B flip ({TODAY})"
print("\n=== Commit + push (from WSL -- Colab has no git creds) ===")
for c in [
    f"cd {REPO_PATH} && git add {' '.join(rel_paths)}",
    f"cd {REPO_PATH} && git commit -m {shlex.quote(msg)}",
    f"cd {REPO_PATH} && git push",
]:
    print(f"  {c}")

print("\nNext: integration notebook (scVI, batch_key=study_id, integrate-then-subset "
      "astro+micro). Carried-forward: Li niche-mito diagnostic must run post-integration.")

=== Audit summary ===
  audit_f_storage: pass
  audit_b_apoe_metadata: partial
  audit_a_vocab_niche_critical: partial
  li2025_qc_trace: qc_trace (n_cells_final=381898)
  haney_qc_trace: qc_trace (n_cells_final=112843)

=== Artifacts ===
  committable:  /content/ad-glia-fm-prep/outputs/software_versions/colab_03_2026-06-10_pip_freeze.txt
  committable:  /content/ad-glia-fm-prep/outputs/software_versions/colab_03_2026-06-10_env.json
  committable:  /content/ad-glia-fm-prep/outputs/audit_report.json
  drive-only:   /content/drive/MyDrive/ad-glia-fm-prep/processed/haney_processed.h5ad

=== Commit + push (from WSL -- Colab has no git creds) ===
  cd /content/ad-glia-fm-prep && git add outputs/software_versions/colab_03_2026-06-10_pip_freeze.txt outputs/software_versions/colab_03_2026-06-10_env.json outputs/audit_report.json
  cd /content/ad-glia-fm-prep && git commit -m 'colab_03: Haney 2024 QC + Audit A (Haney row) + Audit B flip (2026-06-10)'
  cd /content/ad-glia-fm-prep && git push

N

> **Interpretation — handoff summary (8a).**
>
> All three studies QC-complete. `haney_qc_trace n_cells_final=112,843` in the audit report is the post-secondary-QC count (pre-Scrublet); the canonical final N is **108,764** (post-Scrublet, as in `haney_processed.h5ad`). `audit_b_apoe_metadata` and `audit_a_vocab_niche_critical` remain `partial` because SEA-AD and Li entries were written in prior sessions — all individual study audits pass. Carried forward to integration: (1) Li + Haney niche-mito diagnostic (per-cell-type mito check at ceiling, deferred due to absent cell-type labels at QC time); (2) Audit B SEA-AD and Li flip from `skipped` to `pass` once confirmed in the integration audit cell.